# 팀과제 1 - Q3 (팀 4)

2015~2019년 World Happiness Report에서 **5년 동안 행복지수가 매년 상승하거나 적어도 전년도와 같은 나라**를 찾는다.

- 5개 연도 모두 조사된 나라만 대상
- 국가명에 쉼표가 들어간 경우(예: 2017년 `"Hong Kong S.A.R., China"`)가 있어 `split(',')` 대신 `csv.reader`로 파싱

In [10]:
!pip install -q wget pyspark

In [11]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Q3').getOrCreate()
sc = spark.sparkContext
import csv

years = [2015, 2016, 2017, 2018, 2019]
!for y in 2015 2016 2017 2018 2019; do wget -q -O $y.csv "https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World%20Hapiness%20Report/$y.csv"; done
!ls -l *.csv

-rw-r--r-- 1 root root  9407 Oct  9 12:53 2015.csv
-rw-r--r-- 1 root root  9353 Oct  9 12:53 2016.csv
-rw-r--r-- 1 root root 15082 Oct  9 12:53 2017.csv
-rw-r--r-- 1 root root  7726 Oct  9 12:53 2018.csv
-rw-r--r-- 1 root root  7732 Oct  9 12:53 2019.csv


In [12]:
# 1) 연도별 컬럼 이름 확인
for y in years:
    print(y, sc.textFile(f"{y}.csv").first())

2015 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
2016 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
2017 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Generosity,Trust (Government Corruption)
2018 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
2019 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)


In [13]:
# 2) 확인 결과 5개 연도 모두 'Country'가 0번째, 'Happiness Score'가 2번째 컬럼
def load_year(y):
    return sc.textFile(f"{y}.csv") \
        .filter(lambda line: not line.startswith("Country")) \
        .map(lambda line: next(csv.reader([line]))) \
        .map(lambda cols: (cols[0], (y, float(cols[2]))))

all_years = load_year(2015)
for y in years[1:]:
    all_years = all_years.union(load_year(y))

def non_decreasing(scores):
    s = [score for _, score in sorted(scores)]   # 연도순 정렬
    return all(s[i + 1] >= s[i] for i in range(len(s) - 1))

result_rdd = all_years.groupByKey() \
    .mapValues(list) \
    .filter(lambda kv: len(kv[1]) == 5) \
    .filter(lambda kv: non_decreasing(kv[1])) \
    .map(lambda kv: kv[0]) \
    .sortBy(lambda c: c)

result = result_rdd.collect()
print("나라 수:", len(result))
for country in result:
    print(country)

나라 수: 28
Benin
Burkina Faso
Cambodia
Cameroon
Chad
Congo (Brazzaville)
Czech Republic
Dominican Republic
Estonia
Finland
Gabon
Honduras
Hungary
Ivory Coast
Latvia
Malta
Mongolia
Niger
Philippines
Poland
Portugal
Romania
Senegal
Serbia
Slovakia
Syria
Tajikistan
Togo
